# Error Detection Benchmark

Tests the ability to detect and localize errors in reasoning chains.

## Cognitive Science Background

**Error monitoring** is a key executive metacognitive process — the ability to detect mistakes in one's own reasoning (Yeung & Summerfield, 2012). This benchmark presents reasoning chains with embedded errors and measures detection accuracy, localization precision, and confidence calibration.

**Human d':** 1.5–3.0

## Methodology

Math and logic problems are presented with step-by-step solutions. Some solutions are correct; others have deliberate errors injected at specific steps. The model must identify:

1. **Whether** an error exists (binary detection)
2. **Which step** contains the error (localization)
3. **Confidence** in its judgment (calibration)

### Problem Categories

- **MATH:** Arithmetic and algebra problems
- **LOGIC:** Logical deduction problems
- **PROBABILITY:** Probability and combinatorics

## Scoring

Composite of detection F1, error localization accuracy, ECE (Expected Calibration Error), and gamma correlation. Models that detect errors precisely and are well-calibrated in their detection confidence score highest.

### References

Yeung & Summerfield (2012), Dunlosky & Metcalfe (2009)

In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null

import kaggle_benchmarks as kbench


In [ ]:
_HANDCRAFTED_CHAINS = [
    # === CORRECT CHAINS ===
    {
        "id": "C01",
        "problem": "Solve for x: 3x + 7 = 22",
        "steps": [
            "Step 1: Subtract 7 from both sides: 3x = 22 - 7 = 15",
            "Step 2: Divide both sides by 3: x = 15 / 3 = 5",
            "Step 3: Check: 3(5) + 7 = 15 + 7 = 22 ✓",
        ],
        "final_answer": "x = 5",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "C02",
        "problem": "What is the probability of rolling two dice and getting a sum of 7?",
        "steps": [
            "Step 1: Total possible outcomes when rolling two dice = 6 × 6 = 36",
            "Step 2: Favorable outcomes for sum of 7: (1,6), (2,5), (3,4), (4,3), (5,2), (6,1) = 6 outcomes",
            "Step 3: Probability = favorable / total = 6/36 = 1/6",
        ],
        "final_answer": "1/6",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "C03",
        "problem": "If all roses are flowers, and some flowers are red, can we conclude that some roses are red?",
        "steps": [
            "Step 1: Premise 1: All roses are flowers (roses ⊆ flowers)",
            "Step 2: Premise 2: Some flowers are red (flowers ∩ red ≠ ∅)",
            "Step 3: The red flowers could be non-rose flowers (e.g., tulips, poppies)",
            "Step 4: We cannot conclude that any roses are red — the conclusion does not follow",
        ],
        "final_answer": "No, we cannot conclude that some roses are red",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "C04",
        "problem": "Find the area of a triangle with base 12 cm and height 8 cm.",
        "steps": [
            "Step 1: Area formula for a triangle: A = (1/2) × base × height",
            "Step 2: A = (1/2) × 12 × 8",
            "Step 3: A = (1/2) × 96 = 48 cm²",
        ],
        "final_answer": "48 cm²",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "C05",
        "problem": "How many ways can 5 people be seated in a row?",
        "steps": [
            "Step 1: The first seat can be filled by any of 5 people",
            "Step 2: The second seat by any of the remaining 4",
            "Step 3: Continuing: 3, then 2, then 1",
            "Step 4: Total = 5! = 5 × 4 × 3 × 2 × 1 = 120",
        ],
        "final_answer": "120",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },

    # === CHAINS WITH ERRORS ===
    {
        "id": "E01",
        "problem": "Solve for x: 2(x + 3) = 16",
        "steps": [
            "Step 1: Distribute the 2: 2x + 3 = 16",  # ERROR: should be 2x + 6
            "Step 2: Subtract 3 from both sides: 2x = 13",
            "Step 3: Divide by 2: x = 6.5",
            "Step 4: Check: 2(6.5 + 3) = 2(9.5) = 19 ≠ 16, so let me recheck... Actually 2(6.5 + 3) = 19. Hmm, that's close to 16.",
        ],
        "final_answer": "x = 6.5",
        "has_error": True,
        "error_step": 1,
        "error_description": "Distribution error: 2(x+3) should give 2x + 6, not 2x + 3",
        "difficulty": 1,
    },
    {
        "id": "E02",
        "problem": "What is the probability of getting at least one head in 3 coin flips?",
        "steps": [
            "Step 1: P(at least one head) = 1 - P(no heads) = 1 - P(all tails)",
            "Step 2: P(all tails) = (1/2)³ = 1/6",  # ERROR: should be 1/8
            "Step 3: P(at least one head) = 1 - 1/6 = 5/6",
        ],
        "final_answer": "5/6",
        "has_error": True,
        "error_step": 2,
        "error_description": "Calculation error: (1/2)³ = 1/8, not 1/6",
        "difficulty": 1,
    },
    {
        "id": "E03",
        "problem": "If it rains, the ground is wet. The ground is wet. Did it rain?",
        "steps": [
            "Step 1: Premise: If rain → wet ground",
            "Step 2: Observation: The ground is wet",
            "Step 3: Since wet ground always comes from rain, it must have rained",  # ERROR: affirming the consequent
            "Step 4: Therefore, it rained",
        ],
        "final_answer": "Yes, it rained",
        "has_error": True,
        "error_step": 3,
        "error_description": "Affirming the consequent fallacy: the ground could be wet for other reasons (sprinkler, flood, etc.)",
        "difficulty": 2,
    },
    {
        "id": "E04",
        "problem": "Simplify: (x² - 9) / (x - 3)",
        "steps": [
            "Step 1: Factor the numerator: x² - 9 = (x - 3)(x - 3)",  # ERROR: should be (x-3)(x+3)
            "Step 2: Cancel (x - 3): (x - 3)(x - 3) / (x - 3) = x - 3",
            "Step 3: Result: x - 3 (for x ≠ 3)",
        ],
        "final_answer": "x - 3",
        "has_error": True,
        "error_step": 1,
        "error_description": "Factoring error: x² - 9 = (x-3)(x+3), not (x-3)(x-3). The correct simplification is x + 3.",
        "difficulty": 1,
    },
    {
        "id": "E05",
        "problem": "A train travels 120 km in 1.5 hours. Then it travels 80 km in 1 hour. What is the average speed for the entire trip?",
        "steps": [
            "Step 1: Speed for leg 1: 120/1.5 = 80 km/h",
            "Step 2: Speed for leg 2: 80/1 = 80 km/h",
            "Step 3: Average speed = (80 + 80) / 2 = 80 km/h",  # ERROR: should use total distance / total time
        ],
        "final_answer": "80 km/h",
        "has_error": True,
        "error_step": 3,
        "error_description": "Average speed should be total distance / total time = 200/2.5 = 80 km/h. In this case the answer happens to be correct by coincidence, but the method is wrong (averaging speeds is incorrect in general).",
        "difficulty": 2,
    },
    {
        "id": "E06",
        "problem": "How many diagonals does a hexagon have?",
        "steps": [
            "Step 1: Formula for diagonals of an n-gon: n(n-3)/2",
            "Step 2: For hexagon, n = 6: 6(6-3)/2 = 6 × 3/2 = 18/2 = 9",
        ],
        "final_answer": "9",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "E07",
        "problem": "Convert 5/8 to a percentage.",
        "steps": [
            "Step 1: To convert a fraction to a percentage, multiply by 100",
            "Step 2: 5/8 × 100 = 500/8 = 62.5%",
        ],
        "final_answer": "62.5%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "E08",
        "problem": "What is the derivative of f(x) = x³ + 2x² - 5x + 1?",
        "steps": [
            "Step 1: Apply power rule term by term",
            "Step 2: d/dx(x³) = 3x²",
            "Step 3: d/dx(2x²) = 4x",
            "Step 4: d/dx(-5x) = -5",
            "Step 5: d/dx(1) = 1",  # ERROR: derivative of constant is 0
            "Step 6: f'(x) = 3x² + 4x - 5 + 1 = 3x² + 4x - 4",
        ],
        "final_answer": "f'(x) = 3x² + 4x - 4",
        "has_error": True,
        "error_step": 5,
        "error_description": "The derivative of a constant (1) is 0, not 1. Correct answer: f'(x) = 3x² + 4x - 5",
        "difficulty": 2,
    },
    {
        "id": "E09",
        "problem": "A bag contains 3 red and 5 blue balls. Two balls are drawn without replacement. What is the probability both are red?",
        "steps": [
            "Step 1: P(first red) = 3/8",
            "Step 2: After drawing one red, remaining: 2 red, 5 blue = 7 total",
            "Step 3: P(second red | first red) = 2/7",
            "Step 4: P(both red) = (3/8) × (2/7) = 6/56 = 3/28",
        ],
        "final_answer": "3/28",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "E10",
        "problem": "Evaluate: log₂(8) + log₂(4)",
        "steps": [
            "Step 1: log₂(8) = 3 (since 2³ = 8)",
            "Step 2: log₂(4) = 2 (since 2² = 4)",
            "Step 3: By the log addition rule: log₂(8) + log₂(4) = log₂(8 × 4) = log₂(32) = 5",
        ],
        "final_answer": "5",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "E11",
        "problem": "Find the sum of the interior angles of a pentagon.",
        "steps": [
            "Step 1: Formula: sum of interior angles = (n-2) × 180°",
            "Step 2: For pentagon, n = 5: (5-2) × 180° = 3 × 180° = 480°",  # ERROR: 3 × 180 = 540
        ],
        "final_answer": "480°",
        "has_error": True,
        "error_step": 2,
        "error_description": "Arithmetic error: 3 × 180 = 540, not 480",
        "difficulty": 1,
    },
    {
        "id": "E12",
        "problem": "All cats are mammals. All mammals are warm-blooded. Therefore?",
        "steps": [
            "Step 1: Cats ⊆ Mammals (all cats are mammals)",
            "Step 2: Mammals ⊆ Warm-blooded (all mammals are warm-blooded)",
            "Step 3: By transitivity: Cats ⊆ Warm-blooded",
            "Step 4: Therefore, all cats are warm-blooded",
        ],
        "final_answer": "All cats are warm-blooded",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    # Additional error chains for balance (targeting ~50/50 ratio)
    {
        "id": "E13",
        "problem": "What is the probability of drawing two aces in a row from a standard deck (without replacement)?",
        "steps": [
            "Step 1: P(first ace) = 4/52 = 1/13",
            "Step 2: After drawing one ace, 3 aces remain in 51 cards",
            "Step 3: P(second ace | first ace) = 3/52",  # ERROR: should be 3/51
            "Step 4: P(both aces) = (4/52) × (3/52) = 12/2704 = 3/676",
        ],
        "final_answer": "3/676",
        "has_error": True,
        "error_step": 3,
        "error_description": "Should be 3/51 (not 3/52) since one card has been removed. Correct answer: 12/2652 = 1/221",
        "difficulty": 2,
    },
    {
        "id": "E14",
        "problem": "Solve: |2x - 6| = 10",
        "steps": [
            "Step 1: |2x - 6| = 10 means either 2x - 6 = 10 or 2x - 6 = -10",
            "Step 2: Case 1: 2x - 6 = 10 → 2x = 16 → x = 8",
            "Step 3: Case 2: 2x - 6 = -10 → 2x = -4 → x = -2",
            "Step 4: Solutions: x = 8 or x = -2",
        ],
        "final_answer": "x = 8 or x = -2",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "E15",
        "problem": "What is the volume of a sphere with radius 3 cm?",
        "steps": [
            "Step 1: Volume formula: V = (4/3)πr²",  # ERROR: should be r³
            "Step 2: V = (4/3) × π × 3² = (4/3) × π × 9 = 12π",
            "Step 3: V ≈ 12 × 3.14159 ≈ 37.7 cm³",
        ],
        "final_answer": "37.7 cm³",
        "has_error": True,
        "error_step": 1,
        "error_description": "Volume formula should be (4/3)πr³, not (4/3)πr². Correct: (4/3)π(27) = 36π ≈ 113.1 cm³",
        "difficulty": 1,
    },
    {
        "id": "E16",
        "problem": "If the sequence follows the pattern: 2, 6, 18, 54, ... what is the 6th term?",
        "steps": [
            "Step 1: Find the common ratio: 6/2 = 3",
            "Step 2: This is a geometric sequence with a₁ = 2, r = 3",
            "Step 3: General term: aₙ = a₁ × r^(n-1) = 2 × 3^(n-1)",
            "Step 4: a₆ = 2 × 3^5 = 2 × 243 = 486",
        ],
        "final_answer": "486",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },

    # === SUBTLE ERROR CHAINS (difficulty=3) — plausible, easy-to-miss errors ===
    {
        "id": "E17",
        "problem": "How many ways can you choose a committee of 3 from 8 people?",
        "steps": [
            "Step 1: This is a combination problem: C(8,3) = 8! / (3! × 5!)",
            "Step 2: 8! / (3! × 5!) = (8 × 7 × 6) / (3 × 2 × 1)",
            "Step 3: Numerator: 8 × 7 × 6 = 336",
            "Step 4: Denominator: 3 × 2 × 1 = 6",
            "Step 5: 336 / 6 = 56",
        ],
        "final_answer": "56",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E18",
        "problem": "Find the integral of 2x × cos(x²) dx",
        "steps": [
            "Step 1: Let u = x², then du = 2x dx",
            "Step 2: The integral becomes ∫ cos(u) du",
            "Step 3: ∫ cos(u) du = -sin(u) + C",  # ERROR: should be +sin(u)
            "Step 4: Substituting back: -sin(x²) + C",
        ],
        "final_answer": "-sin(x²) + C",
        "has_error": True,
        "error_step": 3,
        "error_description": "Sign error: ∫cos(u)du = sin(u) + C, not -sin(u) + C. Easy to confuse with derivative of sin.",
        "difficulty": 3,
    },
    {
        "id": "E19",
        "problem": "In a group of 30 people, what's the probability that at least 2 share a birthday? (Approximate)",
        "steps": [
            "Step 1: P(at least 2 share) = 1 - P(all different)",
            "Step 2: P(all different) = (365/365) × (364/365) × (363/365) × ... × (336/365)",
            "Step 3: Using the approximation: P(all different) ≈ e^(-n(n-1)/(2×365))",
            "Step 4: n(n-1)/2 = 30 × 29/2 = 435",
            "Step 5: P(all different) ≈ e^(-435/365) ≈ e^(-1.192) ≈ 0.304",
            "Step 6: P(at least 2 share) ≈ 1 - 0.304 = 0.696 ≈ 70%",
        ],
        "final_answer": "≈ 70%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E20",
        "problem": "A fair die is rolled 3 times. What is the probability of getting exactly 2 sixes?",
        "steps": [
            "Step 1: This follows a binomial distribution: P(X=k) = C(n,k) × p^k × (1-p)^(n-k)",
            "Step 2: n=3, k=2, p=1/6",
            "Step 3: C(3,2) = 3",
            "Step 4: P(X=2) = 3 × (1/6)² × (5/6)¹",
            "Step 5: = 3 × (1/36) × (5/6) = 3 × 5/216 = 15/216",  # ERROR: should be 15/216 = 5/72, but he writes...
            "Step 6: = 15/216 = 5/71",  # ERROR: 15/216 = 5/72 not 5/71
        ],
        "final_answer": "5/71",
        "has_error": True,
        "error_step": 6,
        "error_description": "Simplification error: 15/216 = 5/72, not 5/71. An off-by-one in the denominator.",
        "difficulty": 3,
    },
    {
        "id": "E21",
        "problem": "A test has 90% sensitivity and 95% specificity. If 1% of the population has the disease, what's the probability someone who tests positive actually has it?",
        "steps": [
            "Step 1: Using Bayes' theorem: P(D|+) = P(+|D)×P(D) / P(+)",
            "Step 2: P(+|D) = 0.90 (sensitivity), P(D) = 0.01",
            "Step 3: P(+|¬D) = 1 - 0.95 = 0.05 (false positive rate)",
            "Step 4: P(+) = P(+|D)×P(D) + P(+|¬D)×P(¬D) = 0.90×0.01 + 0.05×0.99",
            "Step 5: P(+) = 0.009 + 0.0495 = 0.0585",
            "Step 6: P(D|+) = 0.009 / 0.0585 ≈ 0.154 ≈ 15.4%",
        ],
        "final_answer": "≈ 15.4%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E22",
        "problem": "Prove that √2 is irrational (by contradiction)",
        "steps": [
            "Step 1: Assume √2 is rational, so √2 = a/b where a,b are integers with no common factors",
            "Step 2: Then 2 = a²/b², so a² = 2b²",
            "Step 3: Since a² is even, a must be even. Let a = 2k.",
            "Step 4: Then (2k)² = 2b², so 4k² = 2b², so b² = 2k²",  # correct
            "Step 5: Since b² is even, b must be even",
            "Step 6: But then a and b are both even, contradicting our assumption they share no common factors",
            "Step 7: Therefore √2 is irrational ■",
        ],
        "final_answer": "√2 is irrational",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E23",
        "problem": "What is the limit of (1 + 1/n)^n as n → ∞?",
        "steps": [
            "Step 1: Let L = lim(n→∞) (1 + 1/n)^n",
            "Step 2: Take ln of both sides: ln(L) = lim(n→∞) n × ln(1 + 1/n)",
            "Step 3: As n→∞, ln(1 + 1/n) ≈ 1/n - 1/(2n²) (Taylor expansion)",
            "Step 4: So n × ln(1 + 1/n) ≈ n × (1/n - 1/(2n²)) = 1 - 1/(2n) → 1",
            "Step 5: Therefore ln(L) = 1, so L = e¹ = e ≈ 2.718",
        ],
        "final_answer": "e ≈ 2.718",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E24",
        "problem": "How many distinct permutations of the letters in 'MISSISSIPPI'?",
        "steps": [
            "Step 1: MISSISSIPPI has 11 letters",
            "Step 2: Letter counts: M=1, I=4, S=4, P=2",
            "Step 3: Formula: 11! / (1! × 4! × 4! × 2!)",
            "Step 4: 11! = 39916800",
            "Step 5: Denominator: 1 × 24 × 24 × 2 = 1152",
            "Step 6: 39916800 / 1152 = 34650",  # correct
        ],
        "final_answer": "34650",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E25",
        "problem": "Find the determinant of the matrix [[2, 1, 3], [0, -1, 2], [1, 4, -1]]",
        "steps": [
            "Step 1: Expand along the first row: det = 2×det[[-1,2],[4,-1]] - 1×det[[0,2],[1,-1]] + 3×det[[0,-1],[1,4]]",
            "Step 2: det[[-1,2],[4,-1]] = (-1)(-1) - (2)(4) = 1 - 8 = -7",
            "Step 3: det[[0,2],[1,-1]] = (0)(-1) - (2)(1) = -2",
            "Step 4: det[[0,-1],[1,4]] = (0)(4) - (-1)(1) = 0 + 1 = 1",
            "Step 5: det = 2(-7) - 1(-2) + 3(1) = -14 + 2 + 3 = -9",
        ],
        "final_answer": "-9",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E26",
        "problem": "What is the sum of the infinite geometric series: 3 + 3/2 + 3/4 + 3/8 + ...?",
        "steps": [
            "Step 1: First term a = 3, common ratio r = 1/2",
            "Step 2: Since |r| < 1, the series converges",
            "Step 3: Sum = a / (1 - r) = 3 / (1 - 1/2) = 3 / (1/2)",
            "Step 4: = 3 × 2 = 6",
        ],
        "final_answer": "6",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E27",
        "problem": "Find the derivative of f(x) = ln(sin(x²))",
        "steps": [
            "Step 1: Apply chain rule: f'(x) = (1/sin(x²)) × d/dx[sin(x²)]",
            "Step 2: d/dx[sin(x²)] = cos(x²) × d/dx[x²] = cos(x²) × 2x",
            "Step 3: f'(x) = (1/sin(x²)) × cos(x²) × 2x = 2x × cos(x²)/sin(x²)",
            "Step 4: = 2x × cot(x²)",
        ],
        "final_answer": "2x·cot(x²)",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E28",
        "problem": "Evaluate: ∫₀¹ x × e^(x²) dx",
        "steps": [
            "Step 1: Let u = x², then du = 2x dx, so x dx = du/2",
            "Step 2: When x=0, u=0; when x=1, u=1",
            "Step 3: The integral becomes (1/2) ∫₀¹ e^u du",
            "Step 4: = (1/2) [e^u]₀¹ = (1/2)(e¹ - e⁰) = (1/2)(e - 1)",
            "Step 5: ≈ (1/2)(2.718 - 1) = (1/2)(1.718) ≈ 0.859",
        ],
        "final_answer": "(e-1)/2 ≈ 0.859",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E29",
        "problem": "A group of 5 people sit around a circular table. How many distinct seating arrangements are there?",
        "steps": [
            "Step 1: For circular permutations, we fix one person and arrange the rest",
            "Step 2: Number of arrangements = (n-1)! = 4! = 24",
        ],
        "final_answer": "24",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E30",
        "problem": "Solve the recurrence: T(n) = 2T(n/2) + n, T(1) = 1. What is T(8)?",
        "steps": [
            "Step 1: T(1) = 1",
            "Step 2: T(2) = 2T(1) + 2 = 2(1) + 2 = 4",
            "Step 3: T(4) = 2T(2) + 4 = 2(4) + 4 = 12",
            "Step 4: T(8) = 2T(4) + 8 = 2(12) + 8 = 32",
        ],
        "final_answer": "32",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E31",
        "problem": "If P(A) = 0.3, P(B) = 0.5, and P(A∩B) = 0.2, find P(A|B)",
        "steps": [
            "Step 1: P(A|B) = P(A∩B) / P(B)",
            "Step 2: P(A|B) = 0.2 / 0.5 = 0.4",
            "Step 3: Also check: P(A∪B) = P(A) + P(B) - P(A∩B) = 0.3 + 0.5 - 0.2 = 0.6 ✓",
        ],
        "final_answer": "0.4",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E32",
        "problem": "Find the eigenvalues of the matrix [[3, 1], [0, 2]]",
        "steps": [
            "Step 1: det(A - λI) = 0",
            "Step 2: det([[3-λ, 1], [0, 2-λ]]) = (3-λ)(2-λ) - 0 = 0",
            "Step 3: (3-λ)(2-λ) = 0",
            "Step 4: λ = 3 or λ = 2",
        ],
        "final_answer": "λ = 2, 3",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },

    # === HARD SUBTLE ERROR CHAINS (difficulty=3) — statistical/logical fallacies ===
    {
        "id": "E33",
        "problem": "A city has two taxi companies: Blue (85% of taxis) and Green (15%). A witness identifies a taxi in an accident as Green. Witnesses correctly identify taxi color 80% of the time. What is the probability the taxi was actually Green?",
        "steps": [
            "Step 1: P(Green) = 0.15, P(Blue) = 0.85",
            "Step 2: P(witness says Green | Green) = 0.80",
            "Step 3: P(witness says Green | Blue) = 0.20",
            "Step 4: The witness is 80% reliable, so the probability is 80%",
        ],
        "final_answer": "80%",
        "has_error": True,
        "error_step": 4,
        "error_description": "Base rate neglect: Must apply Bayes' theorem. P(Green|says Green) = (0.80×0.15)/(0.80×0.15 + 0.20×0.85) = 0.12/0.29 ≈ 41.4%, not 80%.",
        "difficulty": 3,
    },
    {
        "id": "E34",
        "problem": "A hospital compared two treatments for kidney stones. Treatment A: 273/350 (78%) for large stones, 87/87 (100%) for small stones. Treatment B: 289/350 (82.6%) for large stones, 81/87 (93.1%) for small stones. Which treatment is better?",
        "steps": [
            "Step 1: Treatment A: large stones 78%, small stones 100%",
            "Step 2: Treatment B: large stones 82.6%, small stones 93.1%",
            "Step 3: Treatment B beats Treatment A in both categories",
            "Step 4: Therefore Treatment B is the better treatment overall",
        ],
        "final_answer": "Treatment B is better",
        "has_error": True,
        "error_step": 4,
        "error_description": "Simpson's paradox: Must check overall rates. A overall: (273+87)/(350+87) = 360/437 ≈ 82.4%. B overall: (289+81)/(350+87) = 370/437 ≈ 84.7%. But the group sizes within each category differ — Treatment A was given to more severe cases. The conclusion that B is better overall doesn't follow from winning both subgroups when group compositions differ.",
        "difficulty": 3,
    },
    {
        "id": "E35",
        "problem": "A farmer needs to build a straight fence 100 meters long with posts every 10 meters. How many fence posts does he need?",
        "steps": [
            "Step 1: Total fence length: 100 meters",
            "Step 2: Distance between posts: 10 meters",
            "Step 3: Number of posts = 100 / 10 = 10 posts",
        ],
        "final_answer": "10 posts",
        "has_error": True,
        "error_step": 3,
        "error_description": "Off-by-one (fence post error): 100m with posts every 10m creates 10 intervals, requiring 11 posts (one at each end). Number of posts = (length/spacing) + 1 = 11.",
        "difficulty": 3,
    },
    {
        "id": "E36",
        "problem": "A rocket burns fuel at 2.5 kg/s and produces 30 kN of thrust. The specific impulse in seconds is thrust / (fuel_flow × g). Calculate Isp (g = 9.81 m/s²).",
        "steps": [
            "Step 1: Thrust = 30 kN = 30,000 N",
            "Step 2: Fuel flow = 2.5 kg/s",
            "Step 3: Isp = thrust / (fuel_flow × g) = 30,000 / (2.5 × 9.81)",
            "Step 4: Denominator: 2.5 × 9.81 = 24.525",
            "Step 5: Isp = 30,000 / 24.525 = 1,223 seconds",
        ],
        "final_answer": "1,223 seconds",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E37",
        "problem": "A car travels from A to B at 60 km/h and returns from B to A at 40 km/h. What is the average speed for the round trip?",
        "steps": [
            "Step 1: Let distance A→B = d km",
            "Step 2: Time A→B = d/60 hours",
            "Step 3: Time B→A = d/40 hours",
            "Step 4: Total distance = 2d, Total time = d/60 + d/40 = 2d/100 + 3d/100 = 5d/100",
            "Step 5: Average speed = 2d / (5d/100) = 2d × 100/(5d) = 200/5 = 40 km/h",
        ],
        "final_answer": "40 km/h",
        "has_error": True,
        "error_step": 4,
        "error_description": "LCD error: d/60 + d/40. LCD is 120, not 100. d/60 = 2d/120, d/40 = 3d/120. Sum = 5d/120. Average = 2d/(5d/120) = 240/5 = 48 km/h, not 40.",
        "difficulty": 3,
    },
    {
        "id": "E38",
        "problem": "In a class, 70% passed math and 80% passed English. What is the minimum percentage that passed both?",
        "steps": [
            "Step 1: P(Math) = 0.70, P(English) = 0.80",
            "Step 2: By inclusion-exclusion: P(M∪E) = P(M) + P(E) - P(M∩E)",
            "Step 3: Since P(M∪E) ≤ 1: 0.70 + 0.80 - P(M∩E) ≤ 1",
            "Step 4: P(M∩E) ≥ 0.50, so at least 50% passed both",
        ],
        "final_answer": "50%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E39",
        "problem": "Three machines produce widgets. Machine A makes 50% of output with 3% defect rate. Machine B makes 30% with 4% defect rate. Machine C makes 20% with 5% defect rate. A defective widget is found. What's the probability it came from Machine A?",
        "steps": [
            "Step 1: P(def) = 0.50×0.03 + 0.30×0.04 + 0.20×0.05 = 0.015 + 0.012 + 0.010 = 0.037",
            "Step 2: P(A|def) = P(def|A)×P(A) / P(def) = 0.03×0.50 / 0.037",
            "Step 3: = 0.015 / 0.037 ≈ 0.405 ≈ 40.5%",
        ],
        "final_answer": "≈ 40.5%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E40",
        "problem": "A survey of 1000 people found that coffee drinkers had a 20% lower rate of heart disease. The study concludes coffee prevents heart disease.",
        "steps": [
            "Step 1: Observational data: coffee drinkers have 20% lower heart disease rate",
            "Step 2: This establishes a negative correlation between coffee drinking and heart disease",
            "Step 3: A 20% reduction is statistically significant with N=1000",
            "Step 4: Therefore, coffee consumption causes a reduction in heart disease risk",
        ],
        "final_answer": "Coffee prevents heart disease",
        "has_error": True,
        "error_step": 4,
        "error_description": "Correlation-causation fallacy: Observational studies cannot establish causation. Confounders (e.g., coffee drinkers may exercise more, have higher income, better healthcare access) are uncontrolled. Only randomized controlled trials can establish causal claims.",
        "difficulty": 3,
    },

    # === HARD STATISTICAL FALLACY CHAINS (difficulty=3) — ceiling reduction ===
    {
        "id": "E45",
        "problem": "A study finds that countries with higher average income have lower average life satisfaction than poorer countries. A researcher concludes that wealthier individuals are less satisfied with life.",
        "steps": [
            "Step 1: Data shows higher national income correlates with lower national life satisfaction",
            "Step 2: National income reflects the income of the individuals within that country",
            "Step 3: Therefore, individuals with higher income have lower life satisfaction",
        ],
        "final_answer": "Wealthier individuals are less satisfied with life",
        "has_error": True,
        "error_step": 3,
        "error_description": "Ecological fallacy: Relationships observed at the aggregate (country) level cannot be directly attributed to individuals. Within each country, wealthier individuals may well be MORE satisfied. The aggregate pattern may reflect confounds like inequality, cultural factors, or reference-group effects.",
        "difficulty": 3,
    },
    {
        "id": "E46",
        "problem": "A hospital study examines the relationship between exercise and diabetes. Among hospitalized patients, those who exercise regularly are just as likely to have diabetes as sedentary patients. The researchers conclude exercise does not protect against diabetes.",
        "steps": [
            "Step 1: Study population: hospitalized patients",
            "Step 2: Among these patients, exercise frequency shows no association with diabetes prevalence",
            "Step 3: The sample is large (N=2,000) and the analysis is properly controlled for age and sex",
            "Step 4: Therefore, exercise does not reduce diabetes risk in the general population",
        ],
        "final_answer": "Exercise does not protect against diabetes",
        "has_error": True,
        "error_step": 4,
        "error_description": "Berkson's paradox (collider bias): Conditioning on hospitalization creates a spurious association. Hospitalized exercisers are there for OTHER serious conditions (e.g., injuries), while hospitalized sedentary people are there for metabolic/cardiovascular reasons including diabetes. Selecting on hospital admission (a collider) distorts the exercise-diabetes relationship. The conclusion cannot generalize to the non-hospitalized population.",
        "difficulty": 3,
    },
    {
        "id": "E47",
        "problem": "A research team tests 20 dietary supplements for their effect on blood pressure. They find that Supplement K shows a statistically significant reduction (p = 0.03). They publish the result for Supplement K.",
        "steps": [
            "Step 1: 20 supplements tested independently against a placebo",
            "Step 2: Each test uses the standard α = 0.05 significance threshold",
            "Step 3: Supplement K yields p = 0.03, which is below 0.05",
            "Step 4: Therefore, Supplement K significantly reduces blood pressure",
        ],
        "final_answer": "Supplement K significantly reduces blood pressure",
        "has_error": True,
        "error_step": 4,
        "error_description": "Multiple comparisons problem (p-hacking): With 20 independent tests at α = 0.05, the expected number of false positives is 20 × 0.05 = 1.0. Finding one 'significant' result out of 20 is entirely expected by chance. A Bonferroni correction would require p < 0.05/20 = 0.0025, which p = 0.03 does not meet. Publishing only the significant result without adjusting for multiple testing is selective reporting.",
        "difficulty": 3,
    },
    {
        "id": "E48",
        "problem": "An analyst studies successful tech startups and finds that 90% of them had offices in Silicon Valley. She concludes that locating a startup in Silicon Valley greatly increases the probability of success.",
        "steps": [
            "Step 1: Sample: 200 successful tech startups (valued > $1B)",
            "Step 2: 180 out of 200 (90%) were based in Silicon Valley",
            "Step 3: Silicon Valley has strong venture capital networks and talent pools",
            "Step 4: Therefore, locating in Silicon Valley substantially increases startup success probability",
        ],
        "final_answer": "Silicon Valley location greatly increases success probability",
        "has_error": True,
        "error_step": 4,
        "error_description": "Survivorship bias: The study only examines successful startups, ignoring the thousands of failed startups also located in Silicon Valley. If 90% of ALL startups (successful and failed) are in Silicon Valley, the 90% success figure tells us nothing about location's causal effect. Without the base rate of Silicon Valley startups among all startups, the conditional probability P(success|SV) cannot be estimated from P(SV|success) alone.",
        "difficulty": 3,
    },
    {
        "id": "E49",
        "problem": "Students who scored in the bottom 10% on a math exam were given a special tutoring program. On the next exam, their average score improved by 15 points. The school board concludes the tutoring program is effective.",
        "steps": [
            "Step 1: Bottom 10% students identified (mean score: 35/100)",
            "Step 2: These students received 4 weeks of intensive tutoring",
            "Step 3: On the follow-up exam, their mean score was 50/100 (a 15-point improvement)",
            "Step 4: The improvement demonstrates the tutoring program's effectiveness",
        ],
        "final_answer": "The tutoring program raised scores by 15 points",
        "has_error": True,
        "error_step": 4,
        "error_description": "Regression to the mean: Students selected for extreme low scores will naturally score closer to the mean on retest, even without any intervention. Some scored low due to bad luck, illness, or random variation. Without a control group of similarly low-scoring students who did NOT receive tutoring, the 15-point improvement cannot be attributed to the program — it may be entirely or partly an artifact of regression to the mean.",
        "difficulty": 3,
    },
    {
        "id": "E50",
        "problem": "A pharmaceutical company runs a clinical trial with 500 participants. They measure the drug's effect on 40 different health biomarkers. They find statistically significant improvements in 3 biomarkers (p < 0.05 each) and report these as the drug's key benefits.",
        "steps": [
            "Step 1: N = 500 participants, randomized controlled trial",
            "Step 2: 40 biomarkers measured pre- and post-treatment",
            "Step 3: 3 out of 40 biomarkers show p < 0.05 improvement",
            "Step 4: The drug has proven benefits on these 3 specific biomarkers",
        ],
        "final_answer": "The drug improves 3 specific biomarkers",
        "has_error": True,
        "error_step": 4,
        "error_description": "Multiple comparisons / outcome switching: Testing 40 biomarkers at α = 0.05, we expect 40 × 0.05 = 2.0 false positives by chance. Finding 3 significant results is barely above the chance expectation. Reporting these 3 as 'key benefits' without pre-registration of primary endpoints or multiple-testing correction (e.g., Bonferroni: p < 0.00125 required) is outcome switching / p-hacking.",
        "difficulty": 3,
    },
    {
        "id": "E51",
        "problem": "Researchers compared recovery rates of two surgical procedures. Procedure A: 600 patients, 510 recovered (85%). Procedure B: 400 patients, 352 recovered (88%). A colleague notes Procedure B is better. However, when split by severity — Mild cases: A recovered 95% (380/400), B recovered 90% (180/200). Severe cases: A recovered 65% (130/200), B recovered 86% (172/200). The researchers conclude Procedure A is actually better because it wins in both subgroups.",
        "steps": [
            "Step 1: Overall: A = 85%, B = 88% → B looks better",
            "Step 2: Mild: A = 95%, B = 90% → A wins",
            "Step 3: Severe: A = 65%, B = 86% → B wins",
            "Step 4: A wins in mild cases, and since we should look at subgroups, A is the better procedure overall",
        ],
        "final_answer": "Procedure A is better overall",
        "has_error": True,
        "error_step": 4,
        "error_description": "Misapplication of Simpson's paradox reasoning: A does NOT win both subgroups — A wins mild (95% vs 90%) but B wins severe (86% vs 65%). The step falsely claims A wins both. B is clearly superior for severe cases by a large margin (21 percentage points). The conclusion reversal is not justified here because B outperforms A in the severe subgroup.",
        "difficulty": 3,
    },

    # === EASY OBVIOUS ERROR CHAINS (difficulty=1) — anchor low end ===
    {
        "id": "E41",
        "problem": "What is 17 + 28?",
        "steps": [
            "Step 1: Add the ones: 7 + 8 = 15, write 5 carry 1",
            "Step 2: Add the tens: 1 + 2 + 1 (carry) = 5",
            "Step 3: Result: 55",
        ],
        "final_answer": "55",
        "has_error": True,
        "error_step": 2,
        "error_description": "Arithmetic error: 1 + 2 + 1 = 4, not 5. The correct answer is 45.",
        "difficulty": 1,
    },
    {
        "id": "E42",
        "problem": "A rectangle has length 8 cm and width 5 cm. What is its area?",
        "steps": [
            "Step 1: Area of rectangle = length + width",
            "Step 2: Area = 8 + 5 = 13 cm²",
        ],
        "final_answer": "13 cm²",
        "has_error": True,
        "error_step": 1,
        "error_description": "Wrong formula: Area = length × width, not length + width. Correct area = 8 × 5 = 40 cm².",
        "difficulty": 1,
    },
    {
        "id": "E43",
        "problem": "What is 50% of 240?",
        "steps": [
            "Step 1: 50% means divide by 2",
            "Step 2: 240 / 2 = 140",
        ],
        "final_answer": "140",
        "has_error": True,
        "error_step": 2,
        "error_description": "Simple arithmetic error: 240 / 2 = 120, not 140.",
        "difficulty": 1,
    },
    {
        "id": "E44",
        "problem": "Convert 3 hours and 45 minutes to minutes.",
        "steps": [
            "Step 1: 3 hours = 3 × 60 = 180 minutes",
            "Step 2: Total = 180 + 45 = 225 minutes",
        ],
        "final_answer": "225 minutes",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
]

# ─── Add procedurally generated chains for contamination resistance ───
PROCEDURAL_REASONING_CHAINS = 
[{'difficulty': 2,
  'error_description': None,
  'error_step': None,
  'final_answer': '2300',
  'has_error': False,
  'id': 'P101',
  'problem': 'What is 92 × 25?',
  'steps': ['Step 1: Break 92 into 90 + 2',
            'Step 2: 90 × 25 = 2250',
            'Step 3: 2 × 25 = 50',
            'Step 4: 2250 + 50 = 2300']},
 {'difficulty': 2,
  'error_description': None,
  'error_step': None,
  'final_answer': '644',
  'has_error': False,
  'id': 'P102',
  'problem': 'What is 14 × 46?',
  'steps': ['Step 1: Break 14 into 10 + 4',
            'Step 2: 10 × 46 = 460',
            'Step 3: 4 × 46 = 184',
            'Step 4: 460 + 184 = 644']},
 {'difficulty': 2,
  'error_description': None,
  'error_step': None,
  'final_answer': '1638',
  'has_error': False,
  'id': 'P103',
  'problem': 'What is 42 × 39?',
  'steps': ['Step 1: Break 42 into 40 + 2',
            'Step 2: 40 × 39 = 1560',
            'Step 3: 2 × 39 = 78',
            'Step 4: 1560 + 78 = 1638']},
 {'difficulty': 2,
  'error_description': None,
  'error_step': None,
  'final_answer': '672',
  'has_error': False,
  'id': 'P104',
  'problem': 'What is 28 × 24?',
  'steps': ['Step 1: Break 28 into 20 + 8',
            'Step 2: 20 × 24 = 480',
            'Step 3: 8 × 24 = 192',
            'Step 4: 480 + 192 = 672']},
 {'difficulty': 2,
  'error_description': 'Arithmetic error: 40 × 17 = 680, not 681',
  'error_step': 2,
  'final_answer': '783',
  'has_error': True,
  'id': 'P105',
  'problem': 'What is 46 × 17?',
  'steps': ['Step 1: Break 46 into 40 + 6',
            'Step 2: 40 × 17 = 681',
            'Step 3: 6 × 17 = 102',
            'Step 4: 681 + 102 = 783']},
 {'difficulty': 2,
  'error_description': 'Arithmetic error: 10 × 13 = 130, not 120',
  'error_step': 2,
  'final_answer': '172',
  'has_error': True,
  'id': 'P106',
  'problem': 'What is 14 × 13?',
  'steps': ['Step 1: Break 14 into 10 + 4',
            'Step 2: 10 × 13 = 120',
            'Step 3: 4 × 13 = 52',
            'Step 4: 120 + 52 = 172']},
 {'difficulty': 2,
  'error_description': 'Arithmetic error: 20 × 26 = 520, not 510',
  'error_step': 2,
  'final_answer': '640',
  'has_error': True,
  'id': 'P107',
  'problem': 'What is 25 × 26?',
  'steps': ['Step 1: Break 25 into 20 + 5',
            'Step 2: 20 × 26 = 510',
            'Step 3: 5 × 26 = 130',
            'Step 4: 510 + 130 = 640']},
 {'difficulty': 2,
  'error_description': 'Arithmetic error: 40 × 24 = 960, not 961',
  'error_step': 2,
  'final_answer': '1129',
  'has_error': True,
  'id': 'P108',
  'problem': 'What is 47 × 24?',
  'steps': ['Step 1: Break 47 into 40 + 7',
            'Step 2: 40 × 24 = 961',
            'Step 3: 7 × 24 = 168',
            'Step 4: 961 + 168 = 1129']},
 {'difficulty': 1,
  'error_description': None,
  'error_step': None,
  'final_answer': '$227.50',
  'has_error': False,
  'id': 'P109',
  'problem': 'An item costs $325. After a 30% discount, what is the price?',
  'steps': ['Step 1: Calculate discount: 30% of $325 = $325 × 30/100 = $97.50',
            'Step 2: Subtract discount: $325 - $97.50 = $227.50']},
 {'difficulty': 1,
  'error_description': None,
  'error_step': None,
  'final_answer': '$527.25',
  'has_error': False,
  'id': 'P110',
  'problem': 'An item costs $703. After a 25% discount, what is the price?',
  'steps': ['Step 1: Calculate discount: 25% of $703 = $703 × 25/100 = $175.75',
            'Step 2: Subtract discount: $703 - $175.75 = $527.25']},
 {'difficulty': 1,
  'error_description': 'Should subtract the discount, not add it. Correct: $106 - $21.20 = $84.80',
  'error_step': 2,
  'final_answer': '$127.20',
  'has_error': True,
  'id': 'P111',
  'problem': 'An item costs $106. After a 20% discount, what is the price?',
  'steps': ['Step 1: Calculate discount: 20% of $106 = $106 × 20/100 = $21.20',
            'Step 2: Apply discount: $106 + $21.20 = $127.20']},
 {'difficulty': 1,
  'error_description': 'Should subtract the discount, not add it. Correct: $814 - $244.20 = '
                       '$569.80',
  'error_step': 2,
  'final_answer': '$1058.20',
  'has_error': True,
  'id': 'P112',
  'problem': 'An item costs $814. After a 30% discount, what is the price?',
  'steps': ['Step 1: Calculate discount: 30% of $814 = $814 × 30/100 = $244.20',
            'Step 2: Apply discount: $814 + $244.20 = $1058.20']},
 {'difficulty': 2,
  'error_description': None,
  'error_step': None,
  'final_answer': '133',
  'has_error': False,
  'id': 'P113',
  'problem': 'Find the sum of the first 7 terms of the arithmetic sequence: 7, 11, 15, ...',
  'steps': ['Step 1: Identify: a₁ = 7, d = 4, n = 7',
            'Step 2: Last term: aₙ = 7 + (7-1)×4 = 7 + 24 = 31',
            'Step 3: Sum = n(a₁ + aₙ)/2 = 7×(7 + 31)/2 = 7×38/2 = 133']},
 {'difficulty': 2,
  'error_description': None,
  'error_step': None,
  'final_answer': '90',
  'has_error': False,
  'id': 'P114',
  'problem': 'Find the sum of the first 6 terms of the arithmetic sequence: 5, 9, 13, ...',
  'steps': ['Step 1: Identify: a₁ = 5, d = 4, n = 6',
            'Step 2: Last term: aₙ = 5 + (6-1)×4 = 5 + 20 = 25',
            'Step 3: Sum = n(a₁ + aₙ)/2 = 6×(5 + 25)/2 = 6×30/2 = 90']},
 {'difficulty': 2,
  'error_description': 'Formula error: aₙ = a₁ + (n-1)d, not a₁ + nd. Should be 3 + 25 = 28',
  'error_step': 2,
  'final_answer': '108',
  'has_error': True,
  'id': 'P115',
  'problem': 'Find the sum of the first 6 terms: 3, 8, 13, ...',
  'steps': ['Step 1: Identify: a₁ = 3, d = 5, n = 6',
            'Step 2: Last term: aₙ = 3 + 6×5 = 3 + 30 = 33',
            'Step 3: Sum = n(a₁ + aₙ)/2 = 6×(3 + 33)/2 = 108']},
 {'difficulty': 2,
  'error_description': 'Formula error: aₙ = a₁ + (n-1)d, not a₁ + nd. Should be 7 + 36 = 43',
  'error_step': 2,
  'final_answer': '270',
  'has_error': True,
  'id': 'P116',
  'problem': 'Find the sum of the first 10 terms: 7, 11, 15, ...',
  'steps': ['Step 1: Identify: a₁ = 7, d = 4, n = 10',
            'Step 2: Last term: aₙ = 7 + 10×4 = 7 + 40 = 47',
            'Step 3: Sum = n(a₁ + aₙ)/2 = 10×(7 + 47)/2 = 270']}]

# Combine: 16 handcrafted + 16 procedural = 32 total
REASONING_CHAINS = _HANDCRAFTED_CHAINS + PROCEDURAL_REASONING_CHAINS


In [ ]:
"""
MetaCog Benchmark 4: Error Detection (Metacognitive Monitoring of Reasoning)

Tests the model's ability to detect errors in step-by-step reasoning chains.
This measures metacognitive monitoring during/after processing — a critical
component of self-correction capability.

Protocol:
1. Present a problem with a worked step-by-step solution
2. Ask model to review the solution and:
   a. Determine if there's an error (binary)
   b. If yes, identify which step contains the error
   c. Rate confidence in the judgment (0-100)
3. Score based on detection accuracy, localization, and confidence calibration

Cognitive Science Basis:
- Yeung & Summerfield (2012): Error monitoring and metacognition
- Nelson & Narens (1990): Monitoring of ongoing cognitive processes
- Related to "debugging" in education research

Metrics:
- Error detection F1 (binary: error present or not)
- Error localization accuracy (correct step identified)
- Confidence calibration (ECE of error detection confidence)
- Signal detection: d' and meta-d' for error detection

Shortcut Resistance:
- Mix of correct and incorrect chains prevents bias
- Errors vary in subtlety (arithmetic, logic, conceptual)
- Some "errors" are actually correct (tests false alarm rate)
- Confidence calibration penalizes overconfident wrong judgments
"""

from dataclasses import dataclass
import numpy as np
import re
import json
# --- Inlined from benchmarks/metacognition/data/error_detection_chains.py ---
"""
Error Detection benchmark reasoning chains dataset.

Contains math/logic problems with step-by-step solutions.
Some solutions are correct; others have deliberate errors injected
at specific steps. The model must identify:
1. Whether an error exists (binary)
2. Which step contains the error
3. Confidence in its judgment

Categories:
- MATH: Arithmetic and algebra problems
- LOGIC: Logical deduction problems  
- PROBABILITY: Probability/combinatorics

Includes both handcrafted chains (for ecological validity) and
procedurally generated chains (for contamination resistance).
"""

_HANDCRAFTED_CHAINS = [
    # === CORRECT CHAINS ===
    {
        "id": "C01",
        "problem": "Solve for x: 3x + 7 = 22",
        "steps": [
            "Step 1: Subtract 7 from both sides: 3x = 22 - 7 = 15",
            "Step 2: Divide both sides by 3: x = 15 / 3 = 5",
            "Step 3: Check: 3(5) + 7 = 15 + 7 = 22 ✓",
        ],
        "final_answer": "x = 5",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "C02",
        "problem": "What is the probability of rolling two dice and getting a sum of 7?",
        "steps": [
            "Step 1: Total possible outcomes when rolling two dice = 6 × 6 = 36",
            "Step 2: Favorable outcomes for sum of 7: (1,6), (2,5), (3,4), (4,3), (5,2), (6,1) = 6 outcomes",
            "Step 3: Probability = favorable / total = 6/36 = 1/6",
        ],
        "final_answer": "1/6",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "C03",
        "problem": "If all roses are flowers, and some flowers are red, can we conclude that some roses are red?",
        "steps": [
            "Step 1: Premise 1: All roses are flowers (roses ⊆ flowers)",
            "Step 2: Premise 2: Some flowers are red (flowers ∩ red ≠ ∅)",
            "Step 3: The red flowers could be non-rose flowers (e.g., tulips, poppies)",
            "Step 4: We cannot conclude that any roses are red — the conclusion does not follow",
        ],
        "final_answer": "No, we cannot conclude that some roses are red",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "C04",
        "problem": "Find the area of a triangle with base 12 cm and height 8 cm.",
        "steps": [
            "Step 1: Area formula for a triangle: A = (1/2) × base × height",
            "Step 2: A = (1/2) × 12 × 8",
            "Step 3: A = (1/2) × 96 = 48 cm²",
        ],
        "final_answer": "48 cm²",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "C05",
        "problem": "How many ways can 5 people be seated in a row?",
        "steps": [
            "Step 1: The first seat can be filled by any of 5 people",
            "Step 2: The second seat by any of the remaining 4",
            "Step 3: Continuing: 3, then 2, then 1",
            "Step 4: Total = 5! = 5 × 4 × 3 × 2 × 1 = 120",
        ],
        "final_answer": "120",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },

    # === CHAINS WITH ERRORS ===
    {
        "id": "E01",
        "problem": "Solve for x: 2(x + 3) = 16",
        "steps": [
            "Step 1: Distribute the 2: 2x + 3 = 16",  # ERROR: should be 2x + 6
            "Step 2: Subtract 3 from both sides: 2x = 13",
            "Step 3: Divide by 2: x = 6.5",
            "Step 4: Check: 2(6.5 + 3) = 2(9.5) = 19 ≠ 16, so let me recheck... Actually 2(6.5 + 3) = 19. Hmm, that's close to 16.",
        ],
        "final_answer": "x = 6.5",
        "has_error": True,
        "error_step": 1,
        "error_description": "Distribution error: 2(x+3) should give 2x + 6, not 2x + 3",
        "difficulty": 1,
    },
    {
        "id": "E02",
        "problem": "What is the probability of getting at least one head in 3 coin flips?",
        "steps": [
            "Step 1: P(at least one head) = 1 - P(no heads) = 1 - P(all tails)",
            "Step 2: P(all tails) = (1/2)³ = 1/6",  # ERROR: should be 1/8
            "Step 3: P(at least one head) = 1 - 1/6 = 5/6",
        ],
        "final_answer": "5/6",
        "has_error": True,
        "error_step": 2,
        "error_description": "Calculation error: (1/2)³ = 1/8, not 1/6",
        "difficulty": 1,
    },
    {
        "id": "E03",
        "problem": "If it rains, the ground is wet. The ground is wet. Did it rain?",
        "steps": [
            "Step 1: Premise: If rain → wet ground",
            "Step 2: Observation: The ground is wet",
            "Step 3: Since wet ground always comes from rain, it must have rained",  # ERROR: affirming the consequent
            "Step 4: Therefore, it rained",
        ],
        "final_answer": "Yes, it rained",
        "has_error": True,
        "error_step": 3,
        "error_description": "Affirming the consequent fallacy: the ground could be wet for other reasons (sprinkler, flood, etc.)",
        "difficulty": 2,
    },
    {
        "id": "E04",
        "problem": "Simplify: (x² - 9) / (x - 3)",
        "steps": [
            "Step 1: Factor the numerator: x² - 9 = (x - 3)(x - 3)",  # ERROR: should be (x-3)(x+3)
            "Step 2: Cancel (x - 3): (x - 3)(x - 3) / (x - 3) = x - 3",
            "Step 3: Result: x - 3 (for x ≠ 3)",
        ],
        "final_answer": "x - 3",
        "has_error": True,
        "error_step": 1,
        "error_description": "Factoring error: x² - 9 = (x-3)(x+3), not (x-3)(x-3). The correct simplification is x + 3.",
        "difficulty": 1,
    },
    {
        "id": "E05",
        "problem": "A train travels 120 km in 1.5 hours. Then it travels 80 km in 1 hour. What is the average speed for the entire trip?",
        "steps": [
            "Step 1: Speed for leg 1: 120/1.5 = 80 km/h",
            "Step 2: Speed for leg 2: 80/1 = 80 km/h",
            "Step 3: Average speed = (80 + 80) / 2 = 80 km/h",  # ERROR: should use total distance / total time
        ],
        "final_answer": "80 km/h",
        "has_error": True,
        "error_step": 3,
        "error_description": "Average speed should be total distance / total time = 200/2.5 = 80 km/h. In this case the answer happens to be correct by coincidence, but the method is wrong (averaging speeds is incorrect in general).",
        "difficulty": 2,
    },
    {
        "id": "E06",
        "problem": "How many diagonals does a hexagon have?",
        "steps": [
            "Step 1: Formula for diagonals of an n-gon: n(n-3)/2",
            "Step 2: For hexagon, n = 6: 6(6-3)/2 = 6 × 3/2 = 18/2 = 9",
        ],
        "final_answer": "9",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "E07",
        "problem": "Convert 5/8 to a percentage.",
        "steps": [
            "Step 1: To convert a fraction to a percentage, multiply by 100",
            "Step 2: 5/8 × 100 = 500/8 = 62.5%",
        ],
        "final_answer": "62.5%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "E08",
        "problem": "What is the derivative of f(x) = x³ + 2x² - 5x + 1?",
        "steps": [
            "Step 1: Apply power rule term by term",
            "Step 2: d/dx(x³) = 3x²",
            "Step 3: d/dx(2x²) = 4x",
            "Step 4: d/dx(-5x) = -5",
            "Step 5: d/dx(1) = 1",  # ERROR: derivative of constant is 0
            "Step 6: f'(x) = 3x² + 4x - 5 + 1 = 3x² + 4x - 4",
        ],
        "final_answer": "f'(x) = 3x² + 4x - 4",
        "has_error": True,
        "error_step": 5,
        "error_description": "The derivative of a constant (1) is 0, not 1. Correct answer: f'(x) = 3x² + 4x - 5",
        "difficulty": 2,
    },
    {
        "id": "E09",
        "problem": "A bag contains 3 red and 5 blue balls. Two balls are drawn without replacement. What is the probability both are red?",
        "steps": [
            "Step 1: P(first red) = 3/8",
            "Step 2: After drawing one red, remaining: 2 red, 5 blue = 7 total",
            "Step 3: P(second red | first red) = 2/7",
            "Step 4: P(both red) = (3/8) × (2/7) = 6/56 = 3/28",
        ],
        "final_answer": "3/28",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "E10",
        "problem": "Evaluate: log₂(8) + log₂(4)",
        "steps": [
            "Step 1: log₂(8) = 3 (since 2³ = 8)",
            "Step 2: log₂(4) = 2 (since 2² = 4)",
            "Step 3: By the log addition rule: log₂(8) + log₂(4) = log₂(8 × 4) = log₂(32) = 5",
        ],
        "final_answer": "5",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "E11",
        "problem": "Find the sum of the interior angles of a pentagon.",
        "steps": [
            "Step 1: Formula: sum of interior angles = (n-2) × 180°",
            "Step 2: For pentagon, n = 5: (5-2) × 180° = 3 × 180° = 480°",  # ERROR: 3 × 180 = 540
        ],
        "final_answer": "480°",
        "has_error": True,
        "error_step": 2,
        "error_description": "Arithmetic error: 3 × 180 = 540, not 480",
        "difficulty": 1,
    },
    {
        "id": "E12",
        "problem": "All cats are mammals. All mammals are warm-blooded. Therefore?",
        "steps": [
            "Step 1: Cats ⊆ Mammals (all cats are mammals)",
            "Step 2: Mammals ⊆ Warm-blooded (all mammals are warm-blooded)",
            "Step 3: By transitivity: Cats ⊆ Warm-blooded",
            "Step 4: Therefore, all cats are warm-blooded",
        ],
        "final_answer": "All cats are warm-blooded",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    # Additional error chains for balance (targeting ~50/50 ratio)
    {
        "id": "E13",
        "problem": "What is the probability of drawing two aces in a row from a standard deck (without replacement)?",
        "steps": [
            "Step 1: P(first ace) = 4/52 = 1/13",
            "Step 2: After drawing one ace, 3 aces remain in 51 cards",
            "Step 3: P(second ace | first ace) = 3/52",  # ERROR: should be 3/51
            "Step 4: P(both aces) = (4/52) × (3/52) = 12/2704 = 3/676",
        ],
        "final_answer": "3/676",
        "has_error": True,
        "error_step": 3,
        "error_description": "Should be 3/51 (not 3/52) since one card has been removed. Correct answer: 12/2652 = 1/221",
        "difficulty": 2,
    },
    {
        "id": "E14",
        "problem": "Solve: |2x - 6| = 10",
        "steps": [
            "Step 1: |2x - 6| = 10 means either 2x - 6 = 10 or 2x - 6 = -10",
            "Step 2: Case 1: 2x - 6 = 10 → 2x = 16 → x = 8",
            "Step 3: Case 2: 2x - 6 = -10 → 2x = -4 → x = -2",
            "Step 4: Solutions: x = 8 or x = -2",
        ],
        "final_answer": "x = 8 or x = -2",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "E15",
        "problem": "What is the volume of a sphere with radius 3 cm?",
        "steps": [
            "Step 1: Volume formula: V = (4/3)πr²",  # ERROR: should be r³
            "Step 2: V = (4/3) × π × 3² = (4/3) × π × 9 = 12π",
            "Step 3: V ≈ 12 × 3.14159 ≈ 37.7 cm³",
        ],
        "final_answer": "37.7 cm³",
        "has_error": True,
        "error_step": 1,
        "error_description": "Volume formula should be (4/3)πr³, not (4/3)πr². Correct: (4/3)π(27) = 36π ≈ 113.1 cm³",
        "difficulty": 1,
    },
    {
        "id": "E16",
        "problem": "If the sequence follows the pattern: 2, 6, 18, 54, ... what is the 6th term?",
        "steps": [
            "Step 1: Find the common ratio: 6/2 = 3",
            "Step 2: This is a geometric sequence with a₁ = 2, r = 3",
            "Step 3: General term: aₙ = a₁ × r^(n-1) = 2 × 3^(n-1)",
            "Step 4: a₆ = 2 × 3^5 = 2 × 243 = 486",
        ],
        "final_answer": "486",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },

    # === SUBTLE ERROR CHAINS (difficulty=3) — plausible, easy-to-miss errors ===
    {
        "id": "E17",
        "problem": "How many ways can you choose a committee of 3 from 8 people?",
        "steps": [
            "Step 1: This is a combination problem: C(8,3) = 8! / (3! × 5!)",
            "Step 2: 8! / (3! × 5!) = (8 × 7 × 6) / (3 × 2 × 1)",
            "Step 3: Numerator: 8 × 7 × 6 = 336",
            "Step 4: Denominator: 3 × 2 × 1 = 6",
            "Step 5: 336 / 6 = 56",
        ],
        "final_answer": "56",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E18",
        "problem": "Find the integral of 2x × cos(x²) dx",
        "steps": [
            "Step 1: Let u = x², then du = 2x dx",
            "Step 2: The integral becomes ∫ cos(u) du",
            "Step 3: ∫ cos(u) du = -sin(u) + C",  # ERROR: should be +sin(u)
            "Step 4: Substituting back: -sin(x²) + C",
        ],
        "final_answer": "-sin(x²) + C",
        "has_error": True,
        "error_step": 3,
        "error_description": "Sign error: ∫cos(u)du = sin(u) + C, not -sin(u) + C. Easy to confuse with derivative of sin.",
        "difficulty": 3,
    },
    {
        "id": "E19",
        "problem": "In a group of 30 people, what's the probability that at least 2 share a birthday? (Approximate)",
        "steps": [
            "Step 1: P(at least 2 share) = 1 - P(all different)",
            "Step 2: P(all different) = (365/365) × (364/365) × (363/365) × ... × (336/365)",
            "Step 3: Using the approximation: P(all different) ≈ e^(-n(n-1)/(2×365))",
            "Step 4: n(n-1)/2 = 30 × 29/2 = 435",
            "Step 5: P(all different) ≈ e^(-435/365) ≈ e^(-1.192) ≈ 0.304",
            "Step 6: P(at least 2 share) ≈ 1 - 0.304 = 0.696 ≈ 70%",
        ],
        "final_answer": "≈ 70%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E20",
        "problem": "A fair die is rolled 3 times. What is the probability of getting exactly 2 sixes?",
        "steps": [
            "Step 1: This follows a binomial distribution: P(X=k) = C(n,k) × p^k × (1-p)^(n-k)",
            "Step 2: n=3, k=2, p=1/6",
            "Step 3: C(3,2) = 3",
            "Step 4: P(X=2) = 3 × (1/6)² × (5/6)¹",
            "Step 5: = 3 × (1/36) × (5/6) = 3 × 5/216 = 15/216",  # ERROR: should be 15/216 = 5/72, but he writes...
            "Step 6: = 15/216 = 5/71",  # ERROR: 15/216 = 5/72 not 5/71
        ],
        "final_answer": "5/71",
        "has_error": True,
        "error_step": 6,
        "error_description": "Simplification error: 15/216 = 5/72, not 5/71. An off-by-one in the denominator.",
        "difficulty": 3,
    },
    {
        "id": "E21",
        "problem": "A test has 90% sensitivity and 95% specificity. If 1% of the population has the disease, what's the probability someone who tests positive actually has it?",
        "steps": [
            "Step 1: Using Bayes' theorem: P(D|+) = P(+|D)×P(D) / P(+)",
            "Step 2: P(+|D) = 0.90 (sensitivity), P(D) = 0.01",
            "Step 3: P(+|¬D) = 1 - 0.95 = 0.05 (false positive rate)",
            "Step 4: P(+) = P(+|D)×P(D) + P(+|¬D)×P(¬D) = 0.90×0.01 + 0.05×0.99",
            "Step 5: P(+) = 0.009 + 0.0495 = 0.0585",
            "Step 6: P(D|+) = 0.009 / 0.0585 ≈ 0.154 ≈ 15.4%",
        ],
        "final_answer": "≈ 15.4%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E22",
        "problem": "Prove that √2 is irrational (by contradiction)",
        "steps": [
            "Step 1: Assume √2 is rational, so √2 = a/b where a,b are integers with no common factors",
            "Step 2: Then 2 = a²/b², so a² = 2b²",
            "Step 3: Since a² is even, a must be even. Let a = 2k.",
            "Step 4: Then (2k)² = 2b², so 4k² = 2b², so b² = 2k²",  # correct
            "Step 5: Since b² is even, b must be even",
            "Step 6: But then a and b are both even, contradicting our assumption they share no common factors",
            "Step 7: Therefore √2 is irrational ■",
        ],
        "final_answer": "√2 is irrational",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E23",
        "problem": "What is the limit of (1 + 1/n)^n as n → ∞?",
        "steps": [
            "Step 1: Let L = lim(n→∞) (1 + 1/n)^n",
            "Step 2: Take ln of both sides: ln(L) = lim(n→∞) n × ln(1 + 1/n)",
            "Step 3: As n→∞, ln(1 + 1/n) ≈ 1/n - 1/(2n²) (Taylor expansion)",
            "Step 4: So n × ln(1 + 1/n) ≈ n × (1/n - 1/(2n²)) = 1 - 1/(2n) → 1",
            "Step 5: Therefore ln(L) = 1, so L = e¹ = e ≈ 2.718",
        ],
        "final_answer": "e ≈ 2.718",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E24",
        "problem": "How many distinct permutations of the letters in 'MISSISSIPPI'?",
        "steps": [
            "Step 1: MISSISSIPPI has 11 letters",
            "Step 2: Letter counts: M=1, I=4, S=4, P=2",
            "Step 3: Formula: 11! / (1! × 4! × 4! × 2!)",
            "Step 4: 11! = 39916800",
            "Step 5: Denominator: 1 × 24 × 24 × 2 = 1152",
            "Step 6: 39916800 / 1152 = 34650",  # correct
        ],
        "final_answer": "34650",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E25",
        "problem": "Find the determinant of the matrix [[2, 1, 3], [0, -1, 2], [1, 4, -1]]",
        "steps": [
            "Step 1: Expand along the first row: det = 2×det[[-1,2],[4,-1]] - 1×det[[0,2],[1,-1]] + 3×det[[0,-1],[1,4]]",
            "Step 2: det[[-1,2],[4,-1]] = (-1)(-1) - (2)(4) = 1 - 8 = -7",
            "Step 3: det[[0,2],[1,-1]] = (0)(-1) - (2)(1) = -2",
            "Step 4: det[[0,-1],[1,4]] = (0)(4) - (-1)(1) = 0 + 1 = 1",
            "Step 5: det = 2(-7) - 1(-2) + 3(1) = -14 + 2 + 3 = -9",
        ],
        "final_answer": "-9",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E26",
        "problem": "What is the sum of the infinite geometric series: 3 + 3/2 + 3/4 + 3/8 + ...?",
        "steps": [
            "Step 1: First term a = 3, common ratio r = 1/2",
            "Step 2: Since |r| < 1, the series converges",
            "Step 3: Sum = a / (1 - r) = 3 / (1 - 1/2) = 3 / (1/2)",
            "Step 4: = 3 × 2 = 6",
        ],
        "final_answer": "6",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E27",
        "problem": "Find the derivative of f(x) = ln(sin(x²))",
        "steps": [
            "Step 1: Apply chain rule: f'(x) = (1/sin(x²)) × d/dx[sin(x²)]",
            "Step 2: d/dx[sin(x²)] = cos(x²) × d/dx[x²] = cos(x²) × 2x",
            "Step 3: f'(x) = (1/sin(x²)) × cos(x²) × 2x = 2x × cos(x²)/sin(x²)",
            "Step 4: = 2x × cot(x²)",
        ],
        "final_answer": "2x·cot(x²)",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E28",
        "problem": "Evaluate: ∫₀¹ x × e^(x²) dx",
        "steps": [
            "Step 1: Let u = x², then du = 2x dx, so x dx = du/2",
            "Step 2: When x=0, u=0; when x=1, u=1",
            "Step 3: The integral becomes (1/2) ∫₀¹ e^u du",
            "Step 4: = (1/2) [e^u]₀¹ = (1/2)(e¹ - e⁰) = (1/2)(e - 1)",
            "Step 5: ≈ (1/2)(2.718 - 1) = (1/2)(1.718) ≈ 0.859",
        ],
        "final_answer": "(e-1)/2 ≈ 0.859",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E29",
        "problem": "A group of 5 people sit around a circular table. How many distinct seating arrangements are there?",
        "steps": [
            "Step 1: For circular permutations, we fix one person and arrange the rest",
            "Step 2: Number of arrangements = (n-1)! = 4! = 24",
        ],
        "final_answer": "24",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E30",
        "problem": "Solve the recurrence: T(n) = 2T(n/2) + n, T(1) = 1. What is T(8)?",
        "steps": [
            "Step 1: T(1) = 1",
            "Step 2: T(2) = 2T(1) + 2 = 2(1) + 2 = 4",
            "Step 3: T(4) = 2T(2) + 4 = 2(4) + 4 = 12",
            "Step 4: T(8) = 2T(4) + 8 = 2(12) + 8 = 32",
        ],
        "final_answer": "32",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E31",
        "problem": "If P(A) = 0.3, P(B) = 0.5, and P(A∩B) = 0.2, find P(A|B)",
        "steps": [
            "Step 1: P(A|B) = P(A∩B) / P(B)",
            "Step 2: P(A|B) = 0.2 / 0.5 = 0.4",
            "Step 3: Also check: P(A∪B) = P(A) + P(B) - P(A∩B) = 0.3 + 0.5 - 0.2 = 0.6 ✓",
        ],
        "final_answer": "0.4",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E32",
        "problem": "Find the eigenvalues of the matrix [[3, 1], [0, 2]]",
        "steps": [
            "Step 1: det(A - λI) = 0",
            "Step 2: det([[3-λ, 1], [0, 2-λ]]) = (3-λ)(2-λ) - 0 = 0",
            "Step 3: (3-λ)(2-λ) = 0",
            "Step 4: λ = 3 or λ = 2",
        ],
        "final_answer": "λ = 2, 3",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },

    # === HARD SUBTLE ERROR CHAINS (difficulty=3) — statistical/logical fallacies ===
    {
        "id": "E33",
        "problem": "A city has two taxi companies: Blue (85% of taxis) and Green (15%). A witness identifies a taxi in an accident as Green. Witnesses correctly identify taxi color 80% of the time. What is the probability the taxi was actually Green?",
        "steps": [
            "Step 1: P(Green) = 0.15, P(Blue) = 0.85",
            "Step 2: P(witness says Green | Green) = 0.80",
            "Step 3: P(witness says Green | Blue) = 0.20",
            "Step 4: The witness is 80% reliable, so the probability is 80%",
        ],
        "final_answer": "80%",
        "has_error": True,
        "error_step": 4,
        "error_description": "Base rate neglect: Must apply Bayes' theorem. P(Green|says Green) = (0.80×0.15)/(0.80×0.15 + 0.20×0.85) = 0.12/0.29 ≈ 41.4%, not 80%.",
        "difficulty": 3,
    },
    {
        "id": "E34",
        "problem": "A hospital compared two treatments for kidney stones. Treatment A: 273/350 (78%) for large stones, 87/87 (100%) for small stones. Treatment B: 289/350 (82.6%) for large stones, 81/87 (93.1%) for small stones. Which treatment is better?",
        "steps": [
            "Step 1: Treatment A: large stones 78%, small stones 100%",
            "Step 2: Treatment B: large stones 82.6%, small stones 93.1%",
            "Step 3: Treatment B beats Treatment A in both categories",
            "Step 4: Therefore Treatment B is the better treatment overall",
        ],
        "final_answer": "Treatment B is better",
        "has_error": True,
        "error_step": 4,
        "error_description": "Simpson's paradox: Must check overall rates. A overall: (273+87)/(350+87) = 360/437 ≈ 82.4%. B overall: (289+81)/(350+87) = 370/437 ≈ 84.7%. But the group sizes within each category differ — Treatment A was given to more severe cases. The conclusion that B is better overall doesn't follow from winning both subgroups when group compositions differ.",
        "difficulty": 3,
    },
    {
        "id": "E35",
        "problem": "A farmer needs to build a straight fence 100 meters long with posts every 10 meters. How many fence posts does he need?",
        "steps": [
            "Step 1: Total fence length: 100 meters",
            "Step 2: Distance between posts: 10 meters",
            "Step 3: Number of posts = 100 / 10 = 10 posts",
        ],
        "final_answer": "10 posts",
        "has_error": True,
        "error_step": 3,
        "error_description": "Off-by-one (fence post error): 100m with posts every 10m creates 10 intervals, requiring 11 posts (one at each end). Number of posts = (length/spacing) + 1 = 11.",
        "difficulty": 3,
    },
    {
        "id": "E36",
        "problem": "A rocket burns fuel at 2.5 kg/s and produces 30 kN of thrust. The specific impulse in seconds is thrust / (fuel_flow × g). Calculate Isp (g = 9.81 m/s²).",
        "steps": [
            "Step 1: Thrust = 30 kN = 30,000 N",
            "Step 2: Fuel flow = 2.5 kg/s",
            "Step 3: Isp = thrust / (fuel_flow × g) = 30,000 / (2.5 × 9.81)",
            "Step 4: Denominator: 2.5 × 9.81 = 24.525",
            "Step 5: Isp = 30,000 / 24.525 = 1,223 seconds",
        ],
        "final_answer": "1,223 seconds",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E37",
        "problem": "A car travels from A to B at 60 km/h and returns from B to A at 40 km/h. What is the average speed for the round trip?",
        "steps": [
            "Step 1: Let distance A→B = d km",
            "Step 2: Time A→B = d/60 hours",
            "Step 3: Time B→A = d/40 hours",
            "Step 4: Total distance = 2d, Total time = d/60 + d/40 = 2d/100 + 3d/100 = 5d/100",
            "Step 5: Average speed = 2d / (5d/100) = 2d × 100/(5d) = 200/5 = 40 km/h",
        ],
        "final_answer": "40 km/h",
        "has_error": True,
        "error_step": 4,
        "error_description": "LCD error: d/60 + d/40. LCD is 120, not 100. d/60 = 2d/120, d/40 = 3d/120. Sum = 5d/120. Average = 2d/(5d/120) = 240/5 = 48 km/h, not 40.",
        "difficulty": 3,
    },
    {
        "id": "E38",
        "problem": "In a class, 70% passed math and 80% passed English. What is the minimum percentage that passed both?",
        "steps": [
            "Step 1: P(Math) = 0.70, P(English) = 0.80",
            "Step 2: By inclusion-exclusion: P(M∪E) = P(M) + P(E) - P(M∩E)",
            "Step 3: Since P(M∪E) ≤ 1: 0.70 + 0.80 - P(M∩E) ≤ 1",
            "Step 4: P(M∩E) ≥ 0.50, so at least 50% passed both",
        ],
        "final_answer": "50%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E39",
        "problem": "Three machines produce widgets. Machine A makes 50% of output with 3% defect rate. Machine B makes 30% with 4% defect rate. Machine C makes 20% with 5% defect rate. A defective widget is found. What's the probability it came from Machine A?",
        "steps": [
            "Step 1: P(def) = 0.50×0.03 + 0.30×0.04 + 0.20×0.05 = 0.015 + 0.012 + 0.010 = 0.037",
            "Step 2: P(A|def) = P(def|A)×P(A) / P(def) = 0.03×0.50 / 0.037",
            "Step 3: = 0.015 / 0.037 ≈ 0.405 ≈ 40.5%",
        ],
        "final_answer": "≈ 40.5%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 3,
    },
    {
        "id": "E40",
        "problem": "A survey of 1000 people found that coffee drinkers had a 20% lower rate of heart disease. The study concludes coffee prevents heart disease.",
        "steps": [
            "Step 1: Observational data: coffee drinkers have 20% lower heart disease rate",
            "Step 2: This establishes a negative correlation between coffee drinking and heart disease",
            "Step 3: A 20% reduction is statistically significant with N=1000",
            "Step 4: Therefore, coffee consumption causes a reduction in heart disease risk",
        ],
        "final_answer": "Coffee prevents heart disease",
        "has_error": True,
        "error_step": 4,
        "error_description": "Correlation-causation fallacy: Observational studies cannot establish causation. Confounders (e.g., coffee drinkers may exercise more, have higher income, better healthcare access) are uncontrolled. Only randomized controlled trials can establish causal claims.",
        "difficulty": 3,
    },

    # === HARD STATISTICAL FALLACY CHAINS (difficulty=3) — ceiling reduction ===
    {
        "id": "E45",
        "problem": "A study finds that countries with higher average income have lower average life satisfaction than poorer countries. A researcher concludes that wealthier individuals are less satisfied with life.",
        "steps": [
            "Step 1: Data shows higher national income correlates with lower national life satisfaction",
            "Step 2: National income reflects the income of the individuals within that country",
            "Step 3: Therefore, individuals with higher income have lower life satisfaction",
        ],
        "final_answer": "Wealthier individuals are less satisfied with life",
        "has_error": True,
        "error_step": 3,
        "error_description": "Ecological fallacy: Relationships observed at the aggregate (country) level cannot be directly attributed to individuals. Within each country, wealthier individuals may well be MORE satisfied. The aggregate pattern may reflect confounds like inequality, cultural factors, or reference-group effects.",
        "difficulty": 3,
    },
    {
        "id": "E46",
        "problem": "A hospital study examines the relationship between exercise and diabetes. Among hospitalized patients, those who exercise regularly are just as likely to have diabetes as sedentary patients. The researchers conclude exercise does not protect against diabetes.",
        "steps": [
            "Step 1: Study population: hospitalized patients",
            "Step 2: Among these patients, exercise frequency shows no association with diabetes prevalence",
            "Step 3: The sample is large (N=2,000) and the analysis is properly controlled for age and sex",
            "Step 4: Therefore, exercise does not reduce diabetes risk in the general population",
        ],
        "final_answer": "Exercise does not protect against diabetes",
        "has_error": True,
        "error_step": 4,
        "error_description": "Berkson's paradox (collider bias): Conditioning on hospitalization creates a spurious association. Hospitalized exercisers are there for OTHER serious conditions (e.g., injuries), while hospitalized sedentary people are there for metabolic/cardiovascular reasons including diabetes. Selecting on hospital admission (a collider) distorts the exercise-diabetes relationship. The conclusion cannot generalize to the non-hospitalized population.",
        "difficulty": 3,
    },
    {
        "id": "E47",
        "problem": "A research team tests 20 dietary supplements for their effect on blood pressure. They find that Supplement K shows a statistically significant reduction (p = 0.03). They publish the result for Supplement K.",
        "steps": [
            "Step 1: 20 supplements tested independently against a placebo",
            "Step 2: Each test uses the standard α = 0.05 significance threshold",
            "Step 3: Supplement K yields p = 0.03, which is below 0.05",
            "Step 4: Therefore, Supplement K significantly reduces blood pressure",
        ],
        "final_answer": "Supplement K significantly reduces blood pressure",
        "has_error": True,
        "error_step": 4,
        "error_description": "Multiple comparisons problem (p-hacking): With 20 independent tests at α = 0.05, the expected number of false positives is 20 × 0.05 = 1.0. Finding one 'significant' result out of 20 is entirely expected by chance. A Bonferroni correction would require p < 0.05/20 = 0.0025, which p = 0.03 does not meet. Publishing only the significant result without adjusting for multiple testing is selective reporting.",
        "difficulty": 3,
    },
    {
        "id": "E48",
        "problem": "An analyst studies successful tech startups and finds that 90% of them had offices in Silicon Valley. She concludes that locating a startup in Silicon Valley greatly increases the probability of success.",
        "steps": [
            "Step 1: Sample: 200 successful tech startups (valued > $1B)",
            "Step 2: 180 out of 200 (90%) were based in Silicon Valley",
            "Step 3: Silicon Valley has strong venture capital networks and talent pools",
            "Step 4: Therefore, locating in Silicon Valley substantially increases startup success probability",
        ],
        "final_answer": "Silicon Valley location greatly increases success probability",
        "has_error": True,
        "error_step": 4,
        "error_description": "Survivorship bias: The study only examines successful startups, ignoring the thousands of failed startups also located in Silicon Valley. If 90% of ALL startups (successful and failed) are in Silicon Valley, the 90% success figure tells us nothing about location's causal effect. Without the base rate of Silicon Valley startups among all startups, the conditional probability P(success|SV) cannot be estimated from P(SV|success) alone.",
        "difficulty": 3,
    },
    {
        "id": "E49",
        "problem": "Students who scored in the bottom 10% on a math exam were given a special tutoring program. On the next exam, their average score improved by 15 points. The school board concludes the tutoring program is effective.",
        "steps": [
            "Step 1: Bottom 10% students identified (mean score: 35/100)",
            "Step 2: These students received 4 weeks of intensive tutoring",
            "Step 3: On the follow-up exam, their mean score was 50/100 (a 15-point improvement)",
            "Step 4: The improvement demonstrates the tutoring program's effectiveness",
        ],
        "final_answer": "The tutoring program raised scores by 15 points",
        "has_error": True,
        "error_step": 4,
        "error_description": "Regression to the mean: Students selected for extreme low scores will naturally score closer to the mean on retest, even without any intervention. Some scored low due to bad luck, illness, or random variation. Without a control group of similarly low-scoring students who did NOT receive tutoring, the 15-point improvement cannot be attributed to the program — it may be entirely or partly an artifact of regression to the mean.",
        "difficulty": 3,
    },
    {
        "id": "E50",
        "problem": "A pharmaceutical company runs a clinical trial with 500 participants. They measure the drug's effect on 40 different health biomarkers. They find statistically significant improvements in 3 biomarkers (p < 0.05 each) and report these as the drug's key benefits.",
        "steps": [
            "Step 1: N = 500 participants, randomized controlled trial",
            "Step 2: 40 biomarkers measured pre- and post-treatment",
            "Step 3: 3 out of 40 biomarkers show p < 0.05 improvement",
            "Step 4: The drug has proven benefits on these 3 specific biomarkers",
        ],
        "final_answer": "The drug improves 3 specific biomarkers",
        "has_error": True,
        "error_step": 4,
        "error_description": "Multiple comparisons / outcome switching: Testing 40 biomarkers at α = 0.05, we expect 40 × 0.05 = 2.0 false positives by chance. Finding 3 significant results is barely above the chance expectation. Reporting these 3 as 'key benefits' without pre-registration of primary endpoints or multiple-testing correction (e.g., Bonferroni: p < 0.00125 required) is outcome switching / p-hacking.",
        "difficulty": 3,
    },
    {
        "id": "E51",
        "problem": "Researchers compared recovery rates of two surgical procedures. Procedure A: 600 patients, 510 recovered (85%). Procedure B: 400 patients, 352 recovered (88%). A colleague notes Procedure B is better. However, when split by severity — Mild cases: A recovered 95% (380/400), B recovered 90% (180/200). Severe cases: A recovered 65% (130/200), B recovered 86% (172/200). The researchers conclude Procedure A is actually better because it wins in both subgroups.",
        "steps": [
            "Step 1: Overall: A = 85%, B = 88% → B looks better",
            "Step 2: Mild: A = 95%, B = 90% → A wins",
            "Step 3: Severe: A = 65%, B = 86% → B wins",
            "Step 4: A wins in mild cases, and since we should look at subgroups, A is the better procedure overall",
        ],
        "final_answer": "Procedure A is better overall",
        "has_error": True,
        "error_step": 4,
        "error_description": "Misapplication of Simpson's paradox reasoning: A does NOT win both subgroups — A wins mild (95% vs 90%) but B wins severe (86% vs 65%). The step falsely claims A wins both. B is clearly superior for severe cases by a large margin (21 percentage points). The conclusion reversal is not justified here because B outperforms A in the severe subgroup.",
        "difficulty": 3,
    },

    # === EASY OBVIOUS ERROR CHAINS (difficulty=1) — anchor low end ===
    {
        "id": "E41",
        "problem": "What is 17 + 28?",
        "steps": [
            "Step 1: Add the ones: 7 + 8 = 15, write 5 carry 1",
            "Step 2: Add the tens: 1 + 2 + 1 (carry) = 5",
            "Step 3: Result: 55",
        ],
        "final_answer": "55",
        "has_error": True,
        "error_step": 2,
        "error_description": "Arithmetic error: 1 + 2 + 1 = 4, not 5. The correct answer is 45.",
        "difficulty": 1,
    },
    {
        "id": "E42",
        "problem": "A rectangle has length 8 cm and width 5 cm. What is its area?",
        "steps": [
            "Step 1: Area of rectangle = length + width",
            "Step 2: Area = 8 + 5 = 13 cm²",
        ],
        "final_answer": "13 cm²",
        "has_error": True,
        "error_step": 1,
        "error_description": "Wrong formula: Area = length × width, not length + width. Correct area = 8 × 5 = 40 cm².",
        "difficulty": 1,
    },
    {
        "id": "E43",
        "problem": "What is 50% of 240?",
        "steps": [
            "Step 1: 50% means divide by 2",
            "Step 2: 240 / 2 = 140",
        ],
        "final_answer": "140",
        "has_error": True,
        "error_step": 2,
        "error_description": "Simple arithmetic error: 240 / 2 = 120, not 140.",
        "difficulty": 1,
    },
    {
        "id": "E44",
        "problem": "Convert 3 hours and 45 minutes to minutes.",
        "steps": [
            "Step 1: 3 hours = 3 × 60 = 180 minutes",
            "Step 2: Total = 180 + 45 = 225 minutes",
        ],
        "final_answer": "225 minutes",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
]

# ─── Add procedurally generated chains for contamination resistance ───
# --- Inlined from benchmarks/metacognition/data/procedural_error_chains.py ---
"""
Procedurally generated error detection chains for contamination resistance.

These reasoning chains use randomly generated numbers and novel problem setups
so they cannot appear in any training corpus. Each chain has a known error
status and error location for objective scoring.
"""

import random


def _generate_procedural_chains(seed=42):
    """Generate ~16 additional reasoning chains with procedural content."""
    rng = random.Random(seed)
    chains = []
    chain_id = 100  # Start from P100 to avoid conflicts

    # ─── Correct chains with novel numbers ───────────────────────────
    for i in range(4):
        a = rng.randint(11, 99)
        b = rng.randint(11, 99)
        product = a * b
        chain_id += 1
        chains.append({
            "id": f"P{chain_id}",
            "problem": f"What is {a} × {b}?",
            "steps": [
                f"Step 1: Break {a} into {a // 10 * 10} + {a % 10}",
                f"Step 2: {a // 10 * 10} × {b} = {(a // 10 * 10) * b}",
                f"Step 3: {a % 10} × {b} = {(a % 10) * b}",
                f"Step 4: {(a // 10 * 10) * b} + {(a % 10) * b} = {product}",
            ],
            "final_answer": str(product),
            "has_error": False,
            "error_step": None,
            "error_description": None,
            "difficulty": 2,
        })

    # ─── Chains with arithmetic errors ───────────────────────────────
    for i in range(4):
        a = rng.randint(12, 50)
        b = rng.randint(12, 50)
        correct_product = a * b
        # Inject an error in the partial product
        wrong_partial = (a // 10 * 10) * b + rng.choice([-10, 10, -1, 1])
        wrong_total = wrong_partial + (a % 10) * b
        chain_id += 1
        chains.append({
            "id": f"P{chain_id}",
            "problem": f"What is {a} × {b}?",
            "steps": [
                f"Step 1: Break {a} into {a // 10 * 10} + {a % 10}",
                f"Step 2: {a // 10 * 10} × {b} = {wrong_partial}",
                f"Step 3: {a % 10} × {b} = {(a % 10) * b}",
                f"Step 4: {wrong_partial} + {(a % 10) * b} = {wrong_total}",
            ],
            "final_answer": str(wrong_total),
            "has_error": True,
            "error_step": 2,
            "error_description": f"Arithmetic error: {a // 10 * 10} × {b} = {(a // 10 * 10) * b}, not {wrong_partial}",
            "difficulty": 2,
        })

    # ─── Correct percentage chains ───────────────────────────────────
    for i in range(2):
        base = rng.randint(100, 900)
        pct = rng.choice([15, 20, 25, 30])
        discount = base * pct / 100
        final = base - discount
        chain_id += 1
        chains.append({
            "id": f"P{chain_id}",
            "problem": f"An item costs ${base}. After a {pct}% discount, what is the price?",
            "steps": [
                f"Step 1: Calculate discount: {pct}% of ${base} = ${base} × {pct}/100 = ${discount:.2f}",
                f"Step 2: Subtract discount: ${base} - ${discount:.2f} = ${final:.2f}",
            ],
            "final_answer": f"${final:.2f}",
            "has_error": False,
            "error_step": None,
            "error_description": None,
            "difficulty": 1,
        })

    # ─── Percentage chains with errors ───────────────────────────────
    for i in range(2):
        base = rng.randint(100, 900)
        pct = rng.choice([15, 20, 25, 30])
        # Error: add instead of subtract
        discount = base * pct / 100
        wrong_final = base + discount  # should be base - discount
        chain_id += 1
        chains.append({
            "id": f"P{chain_id}",
            "problem": f"An item costs ${base}. After a {pct}% discount, what is the price?",
            "steps": [
                f"Step 1: Calculate discount: {pct}% of ${base} = ${base} × {pct}/100 = ${discount:.2f}",
                f"Step 2: Apply discount: ${base} + ${discount:.2f} = ${wrong_final:.2f}",
            ],
            "final_answer": f"${wrong_final:.2f}",
            "has_error": True,
            "error_step": 2,
            "error_description": f"Should subtract the discount, not add it. Correct: ${base} - ${discount:.2f} = ${base - discount:.2f}",
            "difficulty": 1,
        })

    # ─── Correct series sum chains ───────────────────────────────────
    for i in range(2):
        a1 = rng.randint(2, 10)
        d = rng.randint(2, 5)
        n = rng.randint(6, 10)
        an = a1 + (n - 1) * d
        s = n * (a1 + an) // 2
        chain_id += 1
        chains.append({
            "id": f"P{chain_id}",
            "problem": f"Find the sum of the first {n} terms of the arithmetic sequence: {a1}, {a1+d}, {a1+2*d}, ...",
            "steps": [
                f"Step 1: Identify: a₁ = {a1}, d = {d}, n = {n}",
                f"Step 2: Last term: aₙ = {a1} + ({n}-1)×{d} = {a1} + {(n-1)*d} = {an}",
                f"Step 3: Sum = n(a₁ + aₙ)/2 = {n}×({a1} + {an})/2 = {n}×{a1+an}/2 = {s}",
            ],
            "final_answer": str(s),
            "has_error": False,
            "error_step": None,
            "error_description": None,
            "difficulty": 2,
        })

    # ─── Series sum with formula error ───────────────────────────────
    for i in range(2):
        a1 = rng.randint(2, 10)
        d = rng.randint(2, 5)
        n = rng.randint(6, 10)
        an = a1 + (n - 1) * d
        wrong_an = a1 + n * d  # Error: should be (n-1)*d
        wrong_s = n * (a1 + wrong_an) // 2
        chain_id += 1
        chains.append({
            "id": f"P{chain_id}",
            "problem": f"Find the sum of the first {n} terms: {a1}, {a1+d}, {a1+2*d}, ...",
            "steps": [
                f"Step 1: Identify: a₁ = {a1}, d = {d}, n = {n}",
                f"Step 2: Last term: aₙ = {a1} + {n}×{d} = {a1} + {n*d} = {wrong_an}",
                f"Step 3: Sum = n(a₁ + aₙ)/2 = {n}×({a1} + {wrong_an})/2 = {wrong_s}",
            ],
            "final_answer": str(wrong_s),
            "has_error": True,
            "error_step": 2,
            "error_description": f"Formula error: aₙ = a₁ + (n-1)d, not a₁ + nd. Should be {a1} + {(n-1)*d} = {an}",
            "difficulty": 2,
        })

    return chains


PROCEDURAL_REASONING_CHAINS = _generate_procedural_chains()

# --- End inline ---

# Combine: 16 handcrafted + 16 procedural = 32 total
REASONING_CHAINS = _HANDCRAFTED_CHAINS + PROCEDURAL_REASONING_CHAINS

# --- End inline ---


@dataclass
class ErrorReview:
    """Model's review of a reasoning chain."""
    has_error: bool       # Does this chain contain an error?
    error_step: int       # Which step (1-indexed), or 0 if no error
    explanation: str      # Explanation of the error (or why it's correct)
    confidence: int       # 0-100 confidence in the judgment


# ─── Helpers ─────────────────────────────────────────────────────

# ─── Helpers ───────────────────────────────────────────────────────

def _strip_think(text: str) -> str:
    """Remove <think>...</think> tags that some models wrap around output."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def goodman_kruskal_gamma(x: list, y: list) -> float:
    n = len(x)
    concordant = 0
    discordant = 0
    for i in range(n):
        for j in range(i + 1, n):
            x_diff = x[i] - x[j]
            y_diff = y[i] - y[j]
            product = x_diff * y_diff
            if product > 0:
                concordant += 1
            elif product < 0:
                discordant += 1
    denom = concordant + discordant
    return (concordant - discordant) / denom if denom > 0 else 0.0


def compute_ece(confidences: list, accuracies: list, n_bins: int = 5) -> float:
    conf = np.array(confidences) / 100.0
    acc = np.array(accuracies, dtype=float)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    total = len(conf)
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        if i == 0:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf > lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / total) * abs(acc[mask].mean() - conf[mask].mean())
    return round(float(ece), 4)


def compute_f1(tp: int, fp: int, fn: int) -> float:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def compute_dprime(hit_rate: float, false_alarm_rate: float) -> float:
    """Compute d' (signal detection sensitivity) without scipy."""
    # Approximate inverse normal CDF using rational approximation
    # (Abramowitz & Stegun, 1964)
    def norminv(p):
        if p <= 0:
            return -4.0
        if p >= 1:
            return 4.0
        # Rational approximation for central region
        if p < 0.5:
            t = np.sqrt(-2 * np.log(p))
            c0, c1, c2 = 2.515517, 0.802853, 0.010328
            d1, d2, d3 = 1.432788, 0.189269, 0.001308
            return -(t - (c0 + c1 * t + c2 * t**2) / (1 + d1 * t + d2 * t**2 + d3 * t**3))
        else:
            t = np.sqrt(-2 * np.log(1 - p))
            c0, c1, c2 = 2.515517, 0.802853, 0.010328
            d1, d2, d3 = 1.432788, 0.189269, 0.001308
            return t - (c0 + c1 * t + c2 * t**2) / (1 + d1 * t + d2 * t**2 + d3 * t**3)

    hr = min(max(hit_rate, 0.01), 0.99)
    far = min(max(false_alarm_rate, 0.01), 0.99)
    return float(norminv(hr) - norminv(far))


# ─── The Benchmark Task ────────────────────────────────────────────

@kbench.task(name="Error Detection")
def metacog_error_detection(llm) -> float:
    """
    Error Detection Benchmark.

    Model reviews step-by-step reasoning chains and must:
    1. Detect whether an error exists
    2. Localize the error (which step)
    3. Rate confidence

    Score = 0.35 * detection_F1 + 0.25 * localization_accuracy
            + 0.20 * (1 - ECE) + 0.20 * confidence_gamma_norm

    Cognitive Science Basis: Yeung & Summerfield (2012), error monitoring.
    """
    results = []

    for chain in REASONING_CHAINS:
        # Format the chain for review
        steps_text = "\n".join(chain["steps"])
        prompt = (
            f"Review the following step-by-step solution for errors.\n\n"
            f"**Problem:** {chain['problem']}\n\n"
            f"**Solution:**\n{steps_text}\n\n"
            f"**Final answer:** {chain['final_answer']}\n\n"
            f"Carefully check each step. Does this solution contain an error?\n\n"
            f"Respond with ONLY a JSON object:\n"
            f'{{"has_error": true/false, "error_step": <step number or 0>, '
            f'"explanation": "<explain the error or why correct>", '
            f'"confidence": <0-100>}}'
        )

        with kbench.chats.new(f"review_{chain['id']}"):
            raw = llm.prompt(prompt)
            cleaned = _strip_think(raw)
            try:
                parsed = json.loads(re.search(r'\{.*\}', cleaned, re.DOTALL).group())
                pred_has_error = bool(parsed.get("has_error", False))
                pred_step = int(parsed.get("error_step", 0))
                confidence = max(0, min(100, int(parsed.get("confidence", 50))))
                explanation = str(parsed.get("explanation", ""))
            except Exception:
                # Crude fallback: look for keywords
                raw_lower = cleaned.lower()
                pred_has_error = any(w in raw_lower for w in ["error", "mistake", "incorrect", "wrong"])
                pred_step = 0
                confidence = 50
                explanation = cleaned[:200]

        # Score this chain
        actual_has_error = chain["has_error"]
        actual_step = chain["error_step"]

        # Detection correctness
        detection_correct = pred_has_error == actual_has_error

        # Localization correctness (only relevant when error exists and was detected)
        localization_correct = False
        if actual_has_error and pred_has_error and actual_step is not None:
            localization_correct = pred_step == actual_step

        results.append({
            "id": chain["id"],
            "problem": chain["problem"][:60],
            "actual_has_error": actual_has_error,
            "pred_has_error": pred_has_error,
            "actual_step": actual_step,
            "pred_step": pred_step,
            "detection_correct": detection_correct,
            "localization_correct": localization_correct,
            "confidence": confidence,
            "explanation": explanation[:100],
            "difficulty": chain["difficulty"],
        })

    # ── Compute Metrics (difficulty-weighted) ──
    # Difficulty weights: d=1 → 1.0, d=2 → 2.0, d=3 → 3.0
    diff_map = {1: 1.0, 2: 2.0, 3: 3.0}

    # Detection F1 (unweighted for standard metric)
    tp = sum(1 for r in results if r["actual_has_error"] and r["pred_has_error"])
    fp = sum(1 for r in results if not r["actual_has_error"] and r["pred_has_error"])
    fn = sum(1 for r in results if r["actual_has_error"] and not r["pred_has_error"])
    tn = sum(1 for r in results if not r["actual_has_error"] and not r["pred_has_error"])

    # Difficulty-weighted detection accuracy
    weighted_correct = sum(diff_map.get(r["difficulty"], 1.0) for r in results if r["detection_correct"])
    weighted_total = sum(diff_map.get(r["difficulty"], 1.0) for r in results)
    weighted_detection = weighted_correct / weighted_total if weighted_total > 0 else 0

    f1 = compute_f1(tp, fp, fn)

    # Localization accuracy (among correctly detected errors)
    error_chains = [r for r in results if r["actual_has_error"] and r["pred_has_error"]]
    # Difficulty-weighted localization accuracy
    if error_chains:
        loc_weighted = sum(diff_map.get(r["difficulty"], 1.0) for r in error_chains if r["localization_correct"])
        loc_total = sum(diff_map.get(r["difficulty"], 1.0) for r in error_chains)
        localization_acc = loc_weighted / loc_total if loc_total > 0 else 0
    else:
        localization_acc = 0.0

    # Confidence calibration
    confidences = [r["confidence"] for r in results]
    detection_accuracies = [r["detection_correct"] for r in results]
    ece = compute_ece(confidences, detection_accuracies)

    # Confidence-accuracy gamma
    gamma = goodman_kruskal_gamma(confidences, [int(a) for a in detection_accuracies])
    gamma_norm = (gamma + 1) / 2

    # Signal detection (d')
    n_signal = sum(1 for r in results if r["actual_has_error"])
    n_noise = sum(1 for r in results if not r["actual_has_error"])
    hit_rate = tp / n_signal if n_signal > 0 else 0
    false_alarm_rate = fp / n_noise if n_noise > 0 else 0

    try:
        dprime = compute_dprime(hit_rate, false_alarm_rate)
    except Exception:
        dprime = 0.0

    # Composite score — uses weighted detection instead of raw F1 for better discrimination
    score = round(
        0.30 * weighted_detection + 0.10 * f1 + 0.25 * localization_acc + 0.20 * (1 - ece) + 0.15 * gamma_norm,
        4
    )

    # ── Logging ──
    print(f"\n{'='*60}")
    print(f"ERROR DETECTION BENCHMARK RESULTS")
    print(f"{'='*60}")
    print(f"Chains reviewed: {len(REASONING_CHAINS)}")
    print(f"  With errors: {n_signal}")
    print(f"  Without errors: {n_noise}")
    print(f"\n--- Detection Performance ---")
    print(f"True Positives:  {tp}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print(f"True Negatives:  {tn}")
    print(f"Detection F1:    {f1:.4f}")
    print(f"Hit rate:        {hit_rate:.2%}")
    print(f"False alarm:     {false_alarm_rate:.2%}")
    print(f"d' (sensitivity):{dprime:+.3f}")

    print(f"\n--- Localization ---")
    print(f"Correctly localized: {sum(1 for r in error_chains if r['localization_correct'])}/{len(error_chains)}")
    print(f"Localization acc:    {localization_acc:.2%}")

    print(f"\n--- Metacognitive Metrics ---")
    print(f"ECE:             {ece:.4f}")
    print(f"Gamma:           {gamma:+.4f}")
    print(f"Mean confidence: {np.mean(confidences):.1f}%")
    print(f"Composite score: {score:.4f}")

    print(f"\n--- Per-Chain Results ---")
    for r in results:
        det = "✓" if r["detection_correct"] else "✗"
        loc = ""
        if r["actual_has_error"] and r["pred_has_error"]:
            loc = " LOC:✓" if r["localization_correct"] else f" LOC:✗(pred={r['pred_step']},actual={r['actual_step']})"
        err_label = "ERR" if r["actual_has_error"] else "OK "
        print(f"  {det} [{r['confidence']:3d}%] [{err_label}] {r['problem'][:45]}...{loc}")

    return score


# ─── Run ────────────────────────────────────────────────────────────


In [ ]:
metacog_error_detection.run(llm=kbench.llm)
